In [6]:
import pandas as pd

In [7]:
dados_4k7_vin = pd.read_csv("https://raw.githubusercontent.com/import-tiago/FEI/refs/heads/main/MSc/PEL309/Paper/Data/4k7_vin.csv", usecols=[3, 4], header=None)
dados_4k7_vout = pd.read_csv("https://raw.githubusercontent.com/import-tiago/FEI/refs/heads/main/MSc/PEL309/Paper/Data/4k7_vout.csv", usecols=[3, 4], header=None)
dados_4k7_iout = pd.read_csv("https://raw.githubusercontent.com/import-tiago/FEI/refs/heads/main/MSc/PEL309/Paper/Data/4k7_iout.csv", usecols=[1], header=None)


In [8]:
dados_4k7_vin.columns = ["time", "vin"]
dados_4k7_vout.columns = ["time", "vout"]
dados_4k7_iout.columns = ["iout"]

In [ ]:
dados_4k7_vout_filtrado = dados_4k7_vout.query("vout < 50 & vout > -50").copy()

threshold = 5
mask = dados_4k7_vout_filtrado["vout"].abs() > threshold

vout_start = mask.idxmax()
vout_end = mask[::-1].idxmax()

dados_4k7_vout_filtrado = dados_4k7_vout_filtrado.loc[vout_start:vout_end].copy()

ax_vout = dados_4k7_vout_filtrado.plot(
    x="time",
    y="vout",
    ylim=(-55, 55),
    title="FES VOUT (4k7 LOAD)"
)

ax_vout.set_yticks(range(-50, 51, 10));

In [ ]:
dados_4k7_vin_filtrado = dados_4k7_vin.query("vin < 3.3 & vin > 0").copy()

vin_base = 1.65
delta = 0.1

# Inicio: primeiro ponto que varia mais que delta em relacao a base
mask_start = (dados_4k7_vin_filtrado["vin"] - vin_base).abs() > delta
vin_start = mask_start[mask_start].index[0]

# Regiao apos o inicio
dados_pos_start = dados_4k7_vin_filtrado.loc[vin_start:]

# Pico da rampa
peak = dados_pos_start["vin"].idxmax()
vin_peak = dados_pos_start.loc[peak, "vin"]

# Final: primeiro ponto apos o pico que cai mais que delta
dados_pos_peak = dados_4k7_vin_filtrado.loc[peak:]

mask_end = (vin_peak - dados_pos_peak["vin"]) > delta
end_idx = mask_end[mask_end].index[0]

# Remove o primeiro ponto da queda
end_pos = dados_4k7_vin_filtrado.index.get_loc(end_idx)
vin_end = dados_4k7_vin_filtrado.index[end_pos - 1]

dados_4k7_vin_filtrado = dados_4k7_vin_filtrado.loc[vin_start:vin_end].copy()

ax_vin = dados_4k7_vin_filtrado.plot(
    x="time",
    y="vin",
    ylim=(0, 3.5),
    title="FES VIN (4k7 LOAD)"
)

ax_vin.set_yticks([i * 0.5 for i in range(0, 8)]);

dados_4k7_vin_vout_filtrado = pd.merge_asof(
    dados_4k7_vin_filtrado.sort_values("time"),
    dados_4k7_vout_filtrado.sort_values("time"),
    on="time",
    direction="nearest"
)

ax = dados_4k7_vin_vout_filtrado.plot(
    x="time",
    y=["vin", "vout"],
    secondary_y="vout",
    title="FES VIN e VOUT (4k7 LOAD)"
)

ax.set_ylabel("VIN")
ax.right_ax.set_ylabel("VOUT");